# 🌿 TobaccoGuard AI — Perbandingan Transfer Learning
## MobileNetV2 vs AlexNet-style (VGG16)

---

**Tujuan:** Membandingkan dua arsitektur CNN menggunakan metode **Transfer Learning yang identik** (pipeline, augmentasi, hyperparameter, dan split data yang sama persis).

| | Model A | Model B |
|---|---|---|
| **Arsitektur** | MobileNetV2 (Google, 2018) | VGG16 (Oxford, 2014) |
| **Metode** | Transfer Learning (ImageNet) | Transfer Learning (ImageNet) |
| **Karakteristik** | Ringan, Depthwise Separable Conv | Klasik, Standard Conv |
| **Parameter** | ~2.4 Juta (head only) | ~14.7 Juta (head only) |

> **Catatan:** AlexNet tidak tersedia sebagai pretrained model di `tf.keras.applications`. VGG16 dipilih sebagai representasi arsitektur CNN klasik yang setara secara akademis karena keduanya menggunakan Standard Convolution berlapis, dan VGG16 memiliki bobot pretrained ImageNet yang tersedia resmi.

**Kelas Klasifikasi:**
- `0` → **Filler** (Low Quality) — daun cacat/rajangan
- `1` → **Binder** (Medium Quality) — daun pengikat
- `2` → **Wrapper** (High Quality) — daun pembungkus premium

## 📦 Sel 1 — Install Dependensi

In [ ]:
!pip install -q tensorflow scikit-learn matplotlib seaborn pandas pillow
import tensorflow as tf
print(f'TensorFlow versi: {tf.__version__}')
print(f'GPU tersedia: {tf.config.list_physical_devices("GPU")}')

## 📂 Sel 2 — Upload & Extract Dataset

Upload file `ai.zip` dari folder proyek Tobacco Anda, atau mount Google Drive.

In [ ]:
import os

# ====================================================
# PILIHAN A: Upload dari komputer lokal
# ====================================================
# from google.colab import files
# uploaded = files.upload()  # Upload ai.zip
# !unzip -q ai.zip -d /content/tobacco_ai

# ====================================================
# PILIHAN B: Mount Google Drive (jika sudah upload di Drive)
# ====================================================
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/ai.zip /content/
# !unzip -q /content/ai.zip -d /content/tobacco_ai

# ====================================================
# PILIHAN C: Clone dari GitHub (jika ada repo)
# ====================================================
# !git clone https://github.com/YOUR_USERNAME/Tobacco.git /content/Tobacco
# DATASET_ROOT = '/content/Tobacco/ai/dataset'

# ====================================================
# Sesuaikan path dataset di sini:
# ====================================================
DATASET_ROOT = '/content/tobacco_ai/dataset'  # Ganti sesuai hasil ekstrak Anda

# Auto-detect sub-folder 'dataset/dataset'
if os.path.exists(os.path.join(DATASET_ROOT, 'dataset')):
    DATASET_ROOT = os.path.join(DATASET_ROOT, 'dataset')

print(f'Dataset root: {DATASET_ROOT}')
for split in ['train', 'valid', 'test']:
    d = os.path.join(DATASET_ROOT, split)
    if os.path.exists(d):
        files_count = len([f for f in os.listdir(d) if f.lower().endswith(('.jpg','.png','.jpeg'))])
        print(f'  [{split}]: {files_count} gambar')

## ⚙️ Sel 3 — Konfigurasi Global (Identik untuk Kedua Model)

In [ ]:
import numpy as np
import random
import json
import time
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd
from PIL import Image, ImageEnhance
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2, VGG16
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ─── Hyperparameter (IDENTIK untuk kedua model) ───
IMG_SIZE      = (224, 224)
BATCH_SIZE    = 16
EPOCHS_PHASE1 = 10    # Fase 1: Frozen base — latih head saja
EPOCHS_PHASE2 = 10    # Fase 2: Fine-tuning layer atas
LR_PHASE1     = 1e-3
LR_PHASE2     = 1e-4
RANDOM_SEED   = 42

CLASS_NAMES   = ['filler', 'binder', 'wrapper']
NUM_CLASSES   = 3

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print('✅ Konfigurasi global siap.')
print(f'   IMG_SIZE      : {IMG_SIZE}')
print(f'   BATCH_SIZE    : {BATCH_SIZE}')
print(f'   EPOCHS_PHASE1 : {EPOCHS_PHASE1}')
print(f'   EPOCHS_PHASE2 : {EPOCHS_PHASE2}')
print(f'   LR_PHASE1     : {LR_PHASE1}')
print(f'   LR_PHASE2     : {LR_PHASE2}')

## 🔄 Sel 4 — Fungsi Load & Augmentasi Dataset (Sama untuk Keduanya)

In [ ]:
def augment_image(img_pil):
    """Augmentasi simulasi kondisi konveyor: rotasi, flip, brightness, contrast."""
    if random.random() > 0.5:
        img_pil = img_pil.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() > 0.5:
        img_pil = img_pil.transpose(Image.FLIP_TOP_BOTTOM)
    angle = random.choice([0, 90, 180, 270]) + random.randint(-15, 15)
    img_pil = img_pil.rotate(angle, expand=False, resample=Image.BICUBIC)
    img_pil = ImageEnhance.Brightness(img_pil).enhance(random.uniform(0.8, 1.2))
    img_pil = ImageEnhance.Contrast(img_pil).enhance(random.uniform(0.8, 1.2))
    return img_pil


def load_dataset():
    """
    Memuat dataset daun tembakau dari folder DATASET_ROOT.
    Mendukung format _classes.csv dan _annotations.createml.json.
    Oversampling 8x pada kelas Filler (Low Quality) untuk mengatasi class imbalance.
    """
    all_images, all_labels = [], []

    for split in ['train', 'valid', 'test']:
        split_dir = os.path.join(DATASET_ROOT, split)
        if not os.path.exists(split_dir):
            continue

        csv_path  = os.path.join(split_dir, '_classes.csv')
        json_path = os.path.join(split_dir, '_annotations.createml.json')

        # --- Format CSV ---
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            print(f'  [{split}] {len(df)} entri (_classes.csv)')
            for _, row in df.iterrows():
                img_path = os.path.join(split_dir, row['filename'])
                if not os.path.exists(img_path): continue
                active = ' '.join([c.strip() for c in df.columns[1:] if row[c] == 1]).lower()
                if   'filler' in active or 'low'    in active: lbl = 0
                elif 'binder' in active or 'medium' in active: lbl = 1
                elif 'wrapper'in active or 'high'   in active: lbl = 2
                else: continue
                try:
                    with Image.open(img_path) as img:
                        img = img.convert('RGB').resize(IMG_SIZE, Image.Resampling.BILINEAR)
                        arr = np.array(img, dtype=np.float32) / 255.0
                        all_images.append(arr); all_labels.append(lbl)
                        # Targeted oversampling Filler 8x
                        if lbl == 0:
                            for _ in range(8):
                                aug = augment_image(Image.fromarray((arr*255).astype(np.uint8)))
                                all_images.append(np.array(aug, dtype=np.float32)/255.0)
                                all_labels.append(0)
                except Exception: continue

        # --- Format CreateML JSON ---
        elif os.path.exists(json_path):
            with open(json_path) as f: anns = json.load(f)
            print(f'  [{split}] {len(anns)} anotasi (CreateML JSON)')
            LABEL_MAP = {'Low Quality':0,'Medium Quality':1,'High Quality':2}
            for item in anns:
                img_path = os.path.join(split_dir, item.get('image',''))
                if not os.path.exists(img_path) or not item.get('annotations'): continue
                ann   = item['annotations'][0]
                lbl   = LABEL_MAP.get(ann.get('label'))
                if lbl is None: continue
                coord = ann.get('coordinates', {})
                try:
                    with Image.open(img_path) as full:
                        full = full.convert('RGB')
                        w, h = full.size
                        if all(k in coord for k in ['x','y','width','height']):
                            cx,cy,bw,bh = coord['x'],coord['y'],coord['width'],coord['height']
                            x1,y1 = max(0,int(cx-bw/2)), max(0,int(cy-bh/2))
                            x2,y2 = min(w,int(cx+bw/2)), min(h,int(cy+bh/2))
                            if (x2-x1)>20 and (y2-y1)>20: full = full.crop((x1,y1,x2,y2))
                        arr = np.array(full.resize(IMG_SIZE, Image.Resampling.BILINEAR), dtype=np.float32)/255.0
                        all_images.append(arr); all_labels.append(lbl)
                        if lbl == 0:
                            for _ in range(8):
                                aug = augment_image(Image.fromarray((arr*255).astype(np.uint8)))
                                all_images.append(np.array(aug, dtype=np.float32)/255.0)
                                all_labels.append(0)
                except Exception: continue

    counter = Counter(all_labels)
    print('\n📊 Distribusi dataset setelah oversampling:')
    for lbl_id, count in sorted(counter.items()):
        print(f'   Kelas {lbl_id} ({CLASS_NAMES[lbl_id].upper():8s}): {count} sampel')

    return np.array(all_images, dtype=np.float32), np.array(all_labels, dtype=np.int32)


# Load dan split dataset (SATU KALI, dipakai KEDUA MODEL)
print('🔄 Memuat dataset ...')
X, y_raw = load_dataset()

class_weights_arr = compute_class_weight('balanced', classes=np.unique(y_raw), y=y_raw)
CLASS_WEIGHTS = {i: float(class_weights_arr[i]) for i in range(len(class_weights_arr))}
print(f'\n⚖️  Bobot Kelas Seimbang: {CLASS_WEIGHTS}')

y_cat = tf.keras.utils.to_categorical(y_raw, num_classes=NUM_CLASSES)

X_train, X_val, y_train, y_val, y_raw_train, y_raw_val = train_test_split(
    X, y_cat, y_raw, test_size=0.20, random_state=RANDOM_SEED, stratify=y_raw
)
print(f'\n✅ Split selesai — Train: {len(X_train)} | Val: {len(X_val)}')

## 🏗️ Sel 5 — Builder Fungsi Model (Transfer Learning)

Kedua model menggunakan **arsitektur head yang identik**:
```
Base Model (frozen) → GlobalAveragePooling2D → Dropout(0.3) → Dense(256) → Dropout(0.2) → Dense(3, softmax)
```

In [ ]:
def build_transfer_model(base_model, model_name='model'):
    """
    Membangun model Transfer Learning dengan head classifier yang identik.
    Fase 1: Base model di-freeze, hanya head yang dilatih.
    """
    base_model.trainable = False  # Freeze semua layer base

    x = base_model.output
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.3, name='drop1')(x)
    x = layers.Dense(256, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.2, name='drop2')(x)
    output = layers.Dense(NUM_CLASSES, activation='softmax', name='prediction')(x)

    model = Model(inputs=base_model.input, outputs=output, name=model_name)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_PHASE1),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def get_callbacks():
    """Callback identik untuk kedua model."""
    return [
        EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
    ]


def fine_tune_model(model, base_model, unfreeze_last_n=20):
    """
    Fase 2: Fine-tuning — unfreeze N layer teratas base model,
    latih dengan learning rate kecil.
    """
    base_model.trainable = True
    # Freeze semua kecuali N layer terakhir
    for layer in base_model.layers[:-unfreeze_last_n]:
        layer.trainable = False

    trainable_count = sum(1 for l in model.layers if l.trainable)
    print(f'  Fine-tune: {trainable_count} layer aktif dari total {len(model.layers)}')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_PHASE2),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


print('✅ Fungsi builder model siap.')

## 🚀 Sel 6 — Training Model A: MobileNetV2 (Transfer Learning)

In [ ]:
print('=' * 60)
print('  MODEL A: MobileNetV2 — Transfer Learning')
print('=' * 60)

# Load base MobileNetV2 pretrained ImageNet
base_mobilenet = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
print(f'Parameter base MobileNetV2: {base_mobilenet.count_params():,}')

model_mobilenet = build_transfer_model(base_mobilenet, model_name='MobileNetV2_TL')
model_mobilenet.summary(expand_nested=False)

# ── Fase 1: Feature Extraction ──
print(f'\n[Fase 1] Training head saja ({EPOCHS_PHASE1} epoch maks) ...')
t0 = time.time()
hist_mn_p1 = model_mobilenet.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS_PHASE1,
    batch_size=BATCH_SIZE,
    class_weight=CLASS_WEIGHTS,
    callbacks=get_callbacks(),
    verbose=1
)

# ── Fase 2: Fine-tuning ──
print(f'\n[Fase 2] Fine-tuning 30 layer teratas ({EPOCHS_PHASE2} epoch maks) ...')
model_mobilenet = fine_tune_model(model_mobilenet, base_mobilenet, unfreeze_last_n=30)
hist_mn_p2 = model_mobilenet.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS_PHASE2,
    batch_size=BATCH_SIZE,
    class_weight=CLASS_WEIGHTS,
    callbacks=get_callbacks(),
    verbose=1
)
duration_mn = round(time.time() - t0, 1)

# Evaluasi final
loss_mn, acc_mn = model_mobilenet.evaluate(X_val, y_val, verbose=0)
y_pred_mn = np.argmax(model_mobilenet.predict(X_val, verbose=0), axis=1)
f1_mn = f1_score(y_raw_val, y_pred_mn, average='macro')

print(f'\n✅ MobileNetV2 selesai dalam {duration_mn}s')
print(f'   Val Accuracy : {acc_mn*100:.2f}%')
print(f'   Val Loss     : {loss_mn:.4f}')
print(f'   Macro F1     : {f1_mn:.4f}')

# Gabungkan history fase 1 + 2
history_mn = {}
for k in hist_mn_p1.history:
    history_mn[k] = hist_mn_p1.history[k] + hist_mn_p2.history[k]

## 🚀 Sel 7 — Training Model B: VGG16 (Transfer Learning)

In [ ]:
print('=' * 60)
print('  MODEL B: VGG16 (AlexNet-class) — Transfer Learning')
print('=' * 60)
print('  Catatan: VGG16 mewakili arsitektur CNN klasik Standard Convolution')
print('  (serupa AlexNet namun memiliki pretrained ImageNet resmi di Keras)')

# Load base VGG16 pretrained ImageNet
base_vgg = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
print(f'Parameter base VGG16: {base_vgg.count_params():,}')

model_vgg = build_transfer_model(base_vgg, model_name='VGG16_TL')
model_vgg.summary(expand_nested=False)

# ── Fase 1: Feature Extraction ──
print(f'\n[Fase 1] Training head saja ({EPOCHS_PHASE1} epoch maks) ...')
t0 = time.time()
hist_vgg_p1 = model_vgg.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS_PHASE1,
    batch_size=BATCH_SIZE,
    class_weight=CLASS_WEIGHTS,
    callbacks=get_callbacks(),
    verbose=1
)

# ── Fase 2: Fine-tuning ──
print(f'\n[Fase 2] Fine-tuning 4 layer teratas ({EPOCHS_PHASE2} epoch maks) ...')
model_vgg = fine_tune_model(model_vgg, base_vgg, unfreeze_last_n=4)
hist_vgg_p2 = model_vgg.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS_PHASE2,
    batch_size=BATCH_SIZE,
    class_weight=CLASS_WEIGHTS,
    callbacks=get_callbacks(),
    verbose=1
)
duration_vgg = round(time.time() - t0, 1)

# Evaluasi final
loss_vgg, acc_vgg = model_vgg.evaluate(X_val, y_val, verbose=0)
y_pred_vgg = np.argmax(model_vgg.predict(X_val, verbose=0), axis=1)
f1_vgg = f1_score(y_raw_val, y_pred_vgg, average='macro')

print(f'\n✅ VGG16 selesai dalam {duration_vgg}s')
print(f'   Val Accuracy : {acc_vgg*100:.2f}%')
print(f'   Val Loss     : {loss_vgg:.4f}')
print(f'   Macro F1     : {f1_vgg:.4f}')

# Gabungkan history fase 1 + 2
history_vgg = {}
for k in hist_vgg_p1.history:
    history_vgg[k] = hist_vgg_p1.history[k] + hist_vgg_p2.history[k]

## 📊 Sel 8 — Visualisasi Perbandingan Lengkap

In [ ]:
COLORS = {
    'mobilenet': ('#2563EB', '#93C5FD'),   # Biru tua & muda
    'vgg':       ('#B45309', '#FCD34D'),   # Cokelat tua & muda
}

fig = plt.figure(figsize=(20, 20), facecolor='#F8F6F3')
fig.suptitle(
    'TobaccoGuard AI — Perbandingan Transfer Learning\nMobileNetV2 vs VGG16 (AlexNet-class)',
    fontsize=17, fontweight='bold', color='#2C1A0E', y=0.98
)

gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

# ── 1. Kurva Akurasi ──
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(history_mn['accuracy'],     color=COLORS['mobilenet'][0], lw=2, label='MobileNetV2 Train')
ax1.plot(history_mn['val_accuracy'], color=COLORS['mobilenet'][0], lw=2, ls='--', label='MobileNetV2 Val')
ax1.plot(history_vgg['accuracy'],    color=COLORS['vgg'][0], lw=2, label='VGG16 Train')
ax1.plot(history_vgg['val_accuracy'],color=COLORS['vgg'][0], lw=2, ls='--', label='VGG16 Val')
phase1_end = len(hist_mn_p1.history['accuracy'])
ax1.axvline(phase1_end - 0.5, color='gray', ls=':', lw=1.5, label='Fase 1→2 (Fine-tune)')
ax1.set_title('Kurva Akurasi Training & Validasi', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
ax1.set_facecolor('#FAFAF8')

# ── 2. Kurva Loss ──
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(history_mn['loss'],     color=COLORS['mobilenet'][0], lw=2, label='MobileNetV2 Train')
ax2.plot(history_mn['val_loss'], color=COLORS['mobilenet'][0], lw=2, ls='--', label='MobileNetV2 Val')
ax2.plot(history_vgg['loss'],    color=COLORS['vgg'][0], lw=2, label='VGG16 Train')
ax2.plot(history_vgg['val_loss'],color=COLORS['vgg'][0], lw=2, ls='--', label='VGG16 Val')
ax2.axvline(phase1_end - 0.5, color='gray', ls=':', lw=1.5)
ax2.set_title('Kurva Loss Training & Validasi', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
ax2.set_facecolor('#FAFAF8')

# ── 3. Confusion Matrix MobileNetV2 ──
ax3 = fig.add_subplot(gs[1, 0])
cm_mn = confusion_matrix(y_raw_val, y_pred_mn)
sns.heatmap(cm_mn, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax3,
            linewidths=0.5, linecolor='white')
ax3.set_title(f'Confusion Matrix — MobileNetV2\nAcc: {acc_mn*100:.2f}% | F1: {f1_mn:.3f}',
              fontweight='bold')
ax3.set_xlabel('Prediksi'); ax3.set_ylabel('Ground Truth')

# ── 4. Confusion Matrix VGG16 ──
ax4 = fig.add_subplot(gs[1, 1])
cm_vgg = confusion_matrix(y_raw_val, y_pred_vgg)
sns.heatmap(cm_vgg, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax4,
            linewidths=0.5, linecolor='white')
ax4.set_title(f'Confusion Matrix — VGG16\nAcc: {acc_vgg*100:.2f}% | F1: {f1_vgg:.3f}',
              fontweight='bold')
ax4.set_xlabel('Prediksi'); ax4.set_ylabel('Ground Truth')

# ── 5. Bar Chart Perbandingan Metrik ──
ax5 = fig.add_subplot(gs[2, :])
metrics_labels = ['Val Accuracy (%)', 'Val Loss', 'Macro F1-Score', 'Params (Juta)', 'Waktu Latih (s/100)']
mn_vals  = [acc_mn*100,  loss_mn,   f1_mn,   model_mobilenet.count_params()/1e6, duration_mn/100]
vgg_vals = [acc_vgg*100, loss_vgg,  f1_vgg,  model_vgg.count_params()/1e6,       duration_vgg/100]
x_pos = np.arange(len(metrics_labels))
w = 0.35
bars1 = ax5.bar(x_pos - w/2, mn_vals,  width=w, color=COLORS['mobilenet'][0], label='MobileNetV2', alpha=0.9)
bars2 = ax5.bar(x_pos + w/2, vgg_vals, width=w, color=COLORS['vgg'][0],       label='VGG16',       alpha=0.9)
for bar in bars1:
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax5.set_xticks(x_pos)
ax5.set_xticklabels(metrics_labels, fontsize=10)
ax5.set_title('Perbandingan Metrik Kinerja — MobileNetV2 vs VGG16', fontweight='bold', fontsize=13)
ax5.legend(fontsize=11); ax5.grid(axis='y', alpha=0.3)
ax5.set_facecolor('#FAFAF8')

plt.savefig('comparison_result.png', dpi=150, bbox_inches='tight', facecolor='#F8F6F3')
plt.show()
print('✅ Grafik tersimpan sebagai comparison_result.png')

## 📋 Sel 9 — Tabel Ringkasan Perbandingan

In [ ]:
print('\n' + '='*70)
print('  RINGKASAN PERBANDINGAN TRANSFER LEARNING — TOBACCOGUARD AI')
print('='*70)
print(f'{"Parameter":<35} {"MobileNetV2":>15} {"VGG16":>15}')
print('-'*70)
params = [
    ('Val Accuracy',          f'{acc_mn*100:.2f}%',                    f'{acc_vgg*100:.2f}%'),
    ('Val Loss',              f'{loss_mn:.4f}',                         f'{loss_vgg:.4f}'),
    ('Macro F1-Score',        f'{f1_mn:.4f}',                           f'{f1_vgg:.4f}'),
    ('Total Parameter',       f'{model_mobilenet.count_params():,}',    f'{model_vgg.count_params():,}'),
    ('Waktu Latih (detik)',   f'{duration_mn}s',                        f'{duration_vgg}s'),
    ('Epoch Dijalankan',      f'{len(history_mn["accuracy"])}',         f'{len(history_vgg["accuracy"])}'),
    ('Arsitektur',            'Depthwise Separable',                    'Standard Conv (13 layer)'),
    ('Pretrained',            'ImageNet',                               'ImageNet'),
    ('Metode',                'Transfer Learning',                      'Transfer Learning'),
]
for name, val_mn, val_vgg in params:
    print(f'{name:<35} {val_mn:>15} {val_vgg:>15}')
print('='*70)

# Classification Report
print('\n📊 Classification Report — MobileNetV2:')
print(classification_report(y_raw_val, y_pred_mn, target_names=CLASS_NAMES))
print('\n📊 Classification Report — VGG16:')
print(classification_report(y_raw_val, y_pred_vgg, target_names=CLASS_NAMES))

winner = 'MobileNetV2' if acc_mn >= acc_vgg else 'VGG16'
print(f'\n🏆 Model dengan akurasi tertinggi: {winner}')

## 💾 Sel 10 — Simpan Model & Download Hasil

In [ ]:
import os
os.makedirs('models', exist_ok=True)

# Simpan kedua model
model_mobilenet.save('models/tobacco_mobilenetv2_tl.h5')
model_vgg.save('models/tobacco_vgg16_tl.h5')
print('✅ Kedua model tersimpan di folder models/')

# Simpan metadata perbandingan
results = {
    'MobileNetV2': {
        'val_accuracy': round(float(acc_mn), 4),
        'val_loss':     round(float(loss_mn), 4),
        'macro_f1':     round(float(f1_mn), 4),
        'total_params': int(model_mobilenet.count_params()),
        'duration_sec': duration_mn,
        'classification_report': classification_report(
            y_raw_val, y_pred_mn, target_names=CLASS_NAMES, output_dict=True)
    },
    'VGG16': {
        'val_accuracy': round(float(acc_vgg), 4),
        'val_loss':     round(float(loss_vgg), 4),
        'macro_f1':     round(float(f1_vgg), 4),
        'total_params': int(model_vgg.count_params()),
        'duration_sec': duration_vgg,
        'classification_report': classification_report(
            y_raw_val, y_pred_vgg, target_names=CLASS_NAMES, output_dict=True)
    }
}
with open('comparison_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('✅ Metadata tersimpan di comparison_results.json')

# Download ke komputer lokal
from google.colab import files
files.download('comparison_result.png')
files.download('comparison_results.json')
# Uncomment untuk download model:
# files.download('models/tobacco_mobilenetv2_tl.h5')
# files.download('models/tobacco_vgg16_tl.h5')